This script updates the slope and bankfull depth calculations for "tricky" rivers (extremely flat rivers where in original algorithm slope = 0 or rivers where GQBF database is incomplete within the study reach)

In [4]:
"""
Tricky Rivers Hydraulic Geometry Updater

This script updates hydraulic geometry CSVs for rivers with manually-calculated slopes.
It reads the tricky rivers CSV, updates the slope values in existing hydraulic geometry CSVs,
and recalculates BASED depth predictions using the new slopes with original width/discharge values.

Author: Huck Rees
Date: January 29, 2026
"""

import os
import pandas as pd
import xgboost as xgb

# ===== CONFIGURATION =====
# Path to the tricky rivers CSV with manually calculated slopes
tricky_rivers_csv = r"E:\Dissertation\Data\RiverMapping\HydraulicGeometry\TrickyRivers_hydraulicgeom_input_data.csv"

# Working directory (same as used in hydraulic geometry calculator)
working_directory = r"E:\Dissertation\Data"

# ===========================


def load_based_model(working_directory):
    """Load the BASED XGBoost model for depth prediction"""
    model_path = os.path.join(working_directory, 'Gearon_etal_2024', 'based-api', 
                              'based_us_sans_trampush_early_stopping_combat_overfitting.ubj')
    
    if not os.path.isfile(model_path):
        raise FileNotFoundError(f"BASED model file not found: {model_path}")
    
    model = xgb.Booster()
    model.load_model(model_path)
    return model


def predict_depth_based(model, width, slope, discharge):
    """
    Predict bankfull depth using BASED model
    
    Parameters:
    -----------
    model : xgb.Booster
        Loaded BASED model
    width : float
        Channel width in meters
    slope : float
        Channel slope (positive value)
    discharge : float
        Bankfull discharge in m³/s
        
    Returns:
    --------
    float or None
        Predicted depth in meters, or None if inputs invalid
    """
    if width is None or slope is None or discharge is None:
        return None
    
    if width <= 0 or discharge <= 0:
        return None
    
    slope_abs = abs(slope)
    if slope_abs == 0:
        return None
    
    input_data = pd.DataFrame({
        'width': [width],
        'slope': [slope_abs],
        'discharge': [discharge]
    })
    
    dmatrix = xgb.DMatrix(input_data)
    
    try:
        prediction = model.predict(dmatrix)
        depth = float(prediction[0])
        if depth <= 0:
            return None
        return depth
    except:
        return None


def update_river_hydraulic_geometry(river_name, new_slope, working_directory, based_model):
    """
    Update hydraulic geometry CSV for a single river
    
    Parameters:
    -----------
    river_name : str
        Name of the river
    new_slope : float
        New manually-calculated slope value (positive)
    working_directory : str
        Base working directory
    based_model : xgb.Booster
        Loaded BASED model
        
    Returns:
    --------
    bool
        True if successful, False if error
    """
    # Path to existing hydraulic geometry CSV
    csv_path = os.path.join(working_directory, "RiverMapping", "HydraulicGeometry", 
                           river_name, f"{river_name}_hydraulic_geometry.csv")
    
    if not os.path.isfile(csv_path):
        print(f"  ⚠ Warning: Hydraulic geometry CSV not found for {river_name}")
        print(f"     Expected: {csv_path}")
        return False
    
    # Read existing CSV
    df = pd.read_csv(csv_path)
    
    # Store original values for reporting
    original_slopes = df['slope'].copy()
    original_depths = df['BASED_depth_m'].copy()
    
    # Update slope (keep as positive/absolute value)
    df['slope'] = new_slope
    
    # Recalculate BASED depth for each reach using original width/qbf + new slope
    for idx, row in df.iterrows():
        width = row['median_width_m']
        discharge = row['median_qbf_m3s']
        
        new_depth = predict_depth_based(based_model, width, new_slope, discharge)
        df.at[idx, 'BASED_depth_m'] = new_depth
    
    # Save updated CSV
    df.to_csv(csv_path, index=False)
    
    # Report changes
    print(f"  ✓ Updated {river_name}")
    print(f"     Slope: {original_slopes.iloc[0]:.6e} → {new_slope:.6e}")
    if original_depths.iloc[0] is not None and df['BASED_depth_m'].iloc[0] is not None:
        print(f"     Depth: {original_depths.iloc[0]:.3f} m → {df['BASED_depth_m'].iloc[0]:.3f} m")
    else:
        print(f"     Depth: {original_depths.iloc[0]} → {df['BASED_depth_m'].iloc[0]}")
    
    return True


def process_tricky_rivers(tricky_rivers_csv, working_directory):
    """
    Main function to process all tricky rivers
    
    Parameters:
    -----------
    tricky_rivers_csv : str
        Path to CSV with manually calculated slopes
    working_directory : str
        Base working directory
    """
    print("\n" + "="*80)
    print("TRICKY RIVERS SLOPE UPDATER")
    print("="*80)
    print(f"Input CSV: {tricky_rivers_csv}")
    print(f"Working directory: {working_directory}\n")
    
    # Load tricky rivers data
    if not os.path.isfile(tricky_rivers_csv):
        raise FileNotFoundError(f"Tricky rivers CSV not found: {tricky_rivers_csv}")
    
    tricky_df = pd.read_csv(tricky_rivers_csv)
    print(f"Found {len(tricky_df)} rivers to update\n")
    
    # Load BASED model
    print("Loading BASED model...")
    try:
        based_model = load_based_model(working_directory)
        print("✓ BASED model loaded successfully\n")
    except Exception as e:
        print(f"✗ Error loading BASED model: {e}")
        return
    
    # Process each river
    print("="*80)
    print("UPDATING RIVERS")
    print("="*80 + "\n")
    
    success_count = 0
    fail_count = 0
    
    for idx, row in tricky_df.iterrows():
        river_name = row['river_name']
        new_slope = row['Slope']  # Already positive in the CSV
        
        print(f"[{idx+1}/{len(tricky_df)}] Processing {river_name}...")
        
        success = update_river_hydraulic_geometry(river_name, new_slope, 
                                                  working_directory, based_model)
        
        if success:
            success_count += 1
        else:
            fail_count += 1
        
        print()
    
    # Summary
    print("="*80)
    print("SUMMARY")
    print("="*80)
    print(f"Successfully updated: {success_count} rivers")
    if fail_count > 0:
        print(f"Failed to update: {fail_count} rivers")
    print("\n✓ Processing complete!")
    print("="*80 + "\n")


if __name__ == "__main__":
    process_tricky_rivers(tricky_rivers_csv, working_directory)


TRICKY RIVERS SLOPE UPDATER
Input CSV: E:\Dissertation\Data\RiverMapping\HydraulicGeometry\TrickyRivers_hydraulicgeom_input_data.csv
Working directory: E:\Dissertation\Data

Found 32 rivers to update

Loading BASED model...
✓ BASED model loaded successfully

UPDATING RIVERS

[1/32] Processing AmuDarya_Kerki...
  ✓ Updated AmuDarya_Kerki
     Slope: 2.632730e-04 → 2.632730e-04
     Depth: 5.409 m → 5.409 m

[2/32] Processing Amur_Khabarovsk...
  ✓ Updated Amur_Khabarovsk
     Slope: 1.123870e-04 → 1.123870e-04
     Depth: 11.941 m → 11.941 m

[3/32] Processing Amur_Komsomolsk...
  ✓ Updated Amur_Komsomolsk
     Slope: 1.319650e-04 → 1.319650e-04
     Depth: 12.735 m → 12.735 m

[4/32] Processing Brahmaputra_Bahadurabad...
  ✓ Updated Brahmaputra_Bahadurabad
     Slope: 3.278690e-04 → 3.278690e-04
     Depth: 10.319 m → 10.319 m

[5/32] Processing Brahmaputra_Pasighat...
  ✓ Updated Brahmaputra_Pasighat
     Slope: 7.621950e-04 → 7.621950e-04
     Depth: 3.903 m → 3.903 m

[6/32] Proces

This script compiles all hydraulic geometry data into a single sheet that can be copy/pasted into the master data table.

In [5]:
"""
Hydraulic Geometry Data Compiler

This script compiles all hydraulic geometry metrics from individual river CSVs
into a single master CSV file with one row per reach.

Extracts: river_name, ds_order, median_width_m, median_qbf_m3s, slope, BASED_depth_m

Author: Huck Rees
Date: January 29, 2026
"""

import os
import pandas as pd

# ===== CONFIGURATION =====
# Working directory containing RiverMapping/HydraulicGeometry folder
working_directory = r"E:\Dissertation\Data"

# Output path for compiled CSV
output_csv = os.path.join(working_directory, "RiverMapping", "HydraulicGeometry", 
                         "compiled_hydraulic_geometry.csv")

# ===========================


def compile_hydraulic_geometry(working_directory, output_csv):
    """
    Compile all hydraulic geometry CSVs into a single master file
    
    Parameters:
    -----------
    working_directory : str
        Base working directory
    output_csv : str
        Path for output CSV file
    """
    hydraulic_geom_dir = os.path.join(working_directory, "RiverMapping", "HydraulicGeometry")
    
    if not os.path.exists(hydraulic_geom_dir):
        raise FileNotFoundError(f"HydraulicGeometry directory not found: {hydraulic_geom_dir}")
    
    print("\n" + "="*80)
    print("HYDRAULIC GEOMETRY DATA COMPILER")
    print("="*80)
    print(f"Source directory: {hydraulic_geom_dir}")
    print(f"Output CSV: {output_csv}\n")
    
    # List to store all data
    all_data = []
    
    # Counter for tracking
    river_count = 0
    reach_count = 0
    skipped_count = 0
    
    # Iterate through all subdirectories
    for river_name in sorted(os.listdir(hydraulic_geom_dir)):
        river_path = os.path.join(hydraulic_geom_dir, river_name)
        
        # Skip if not a directory
        if not os.path.isdir(river_path):
            continue
        
        # Look for the hydraulic geometry CSV
        csv_filename = f"{river_name}_hydraulic_geometry.csv"
        csv_path = os.path.join(river_path, csv_filename)
        
        if not os.path.isfile(csv_path):
            print(f"⚠ Skipping {river_name}: CSV not found")
            skipped_count += 1
            continue
        
        try:
            # Read the CSV
            df = pd.read_csv(csv_path)
            
            # Add river_name column
            df['river_name'] = river_name
            
            # Select and reorder columns
            columns_to_extract = [
                'river_name',
                'ds_order',
                'median_width_m',
                'median_qbf_m3s',
                'slope',
                'BASED_depth_m'
            ]
            
            # Check if all required columns exist
            missing_cols = [col for col in columns_to_extract if col not in df.columns]
            if missing_cols:
                print(f"⚠ Skipping {river_name}: Missing columns {missing_cols}")
                skipped_count += 1
                continue
            
            df_selected = df[columns_to_extract].copy()
            
            # Append to master list
            all_data.append(df_selected)
            
            river_count += 1
            reach_count += len(df_selected)
            
            print(f"✓ Processed {river_name}: {len(df_selected)} reach(es)")
            
        except Exception as e:
            print(f"✗ Error processing {river_name}: {e}")
            skipped_count += 1
            continue
    
    # Combine all data
    if all_data:
        print(f"\n{'='*80}")
        print("COMPILING DATA")
        print("="*80)
        
        master_df = pd.concat(all_data, ignore_index=True)
        
        # Sort by river name and reach order
        master_df = master_df.sort_values(['river_name', 'ds_order']).reset_index(drop=True)
        
        # Save to CSV
        master_df.to_csv(output_csv, index=False)
        
        print(f"✓ Saved compiled data to: {output_csv}")
        print(f"\n{'='*80}")
        print("SUMMARY")
        print("="*80)
        print(f"Total rivers processed: {river_count}")
        print(f"Total reaches compiled: {reach_count}")
        if skipped_count > 0:
            print(f"Rivers skipped: {skipped_count}")
        print(f"\nOutput file: {output_csv}")
        print(f"Total rows: {len(master_df)}")
        print(f"Columns: {', '.join(master_df.columns)}")
        print("="*80 + "\n")
        
        # Display first few rows as preview
        print("Preview of compiled data:")
        print(master_df.head(10).to_string(index=False))
        print("\n...")
        
    else:
        print("\n✗ No data found to compile!")


if __name__ == "__main__":
    compile_hydraulic_geometry(working_directory, output_csv)


HYDRAULIC GEOMETRY DATA COMPILER
Source directory: E:\Dissertation\Data\RiverMapping\HydraulicGeometry
Output CSV: E:\Dissertation\Data\RiverMapping\HydraulicGeometry\compiled_hydraulic_geometry.csv

✓ Processed Aladan_VerkhoyanskiyPerevoz: 1 reach(es)
✓ Processed Amazonas_Jatuarana: 1 reach(es)
✓ Processed Amazonas_Tamshiyacu: 1 reach(es)
✓ Processed AmuDarya_Kerki: 1 reach(es)
✓ Processed Amur_Khabarovsk: 1 reach(es)
✓ Processed Amur_Komsomolsk: 1 reach(es)
✓ Processed Amyl_Kachulka: 1 reach(es)
✓ Processed Apalachicola_NearBlountstown: 1 reach(es)
✓ Processed Araguaia_Aruana: 1 reach(es)
✓ Processed Araguaia_LuizAlves: 1 reach(es)
✓ Processed Araguaia_SaoFelixDoAraguaia: 1 reach(es)
✓ Processed Beas_MandiPlain: 1 reach(es)
✓ Processed Beni_Rurrenabaque: 1 reach(es)
✓ Processed Benue_Ibi: 1 reach(es)
✓ Processed Benue_Umaisha: 1 reach(es)
✓ Processed Bermejo: 20 reach(es)
✓ Processed Bhareli_NTRoadCrossing: 1 reach(es)
✓ Processed BolshayaKet_Rodyonovka: 1 reach(es)
✓ Processed Brah